# Kết hợp Community Detection với TGAT

Notebook 04 cho thấy DGraphFin có cấu trúc community rõ và fraud tập trung nhiều hơn ở một số community. Tuy nhiên, kết quả EDA đó chưa cho biết thông tin community có bổ sung giá trị cho mô hình dự đoán hay không.

Notebook này dùng TGAT — mô hình tốt nhất từ Sprint 3 — làm baseline để kiểm tra riêng hai nhóm thông tin được tạo từ Community Detection:

- **Structural features:** mô tả kích thước, mật độ và mức độ kết nối trong/ngoài community.
- **Community-risk:** fraud rate của community được tính chỉ từ nhãn train.

Bốn biến thể được so sánh để tách rõ đóng góp của từng nhóm thông tin:

| Biến thể | Nội dung | Câu hỏi cần trả lời |
|---|---|---|
| A | TGAT gốc | Baseline đạt kết quả bao nhiêu? |
| B | TGAT + structural features | Cấu trúc community có bổ sung tín hiệu không? |
| C | TGAT + community-risk | Mức độ tập trung fraud theo community có bổ sung tín hiệu không? |
| D | TGAT + structural features + community-risk | Kết hợp cả hai có tốt hơn dùng riêng từng nhóm không? |

A/B/C/D dùng cùng các seed `42/43/44` và checkpoint được chọn bằng validation AP. Tập test chỉ được dùng cho lần đánh giá cuối sau khi cấu hình đã khóa. Để tránh label leakage, community-risk không tham gia quá trình TGAT tổng hợp hàng xóm; risk chỉ được nối vào biểu diễn của node đang cần dự đoán ngay trước classifier.


In [1]:
from __future__ import annotations
from copy import deepcopy
from datetime import datetime
from pathlib import Path
import gc
import hashlib
import json
import os
import platform
import statistics
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root")

PROJECT_ROOT = find_project_root(Path.cwd())
TRAINING_NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "02_gnn_training.ipynb"
DATA_PATH = PROJECT_ROOT / "data" / "dgraphfin.npz"
FEATURE_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_features.npz"
FEATURE_MANIFEST_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_features.manifest.json"
BASELINE_RUN = PROJECT_ROOT / "artifacts" / "runs" / "tgat_undirected_20260812_214306"
RUN_ROOT = PROJECT_ROOT / "artifacts" / "runs"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures" / "sprint4"
RUN_COMMUNITY_ABLATION_TRAIN = os.getenv("RUN_COMMUNITY_ABLATION_TRAIN", "0") == "1"
RUN_COMMUNITY_ABLATION_TEST = os.getenv("RUN_COMMUNITY_ABLATION_TEST", "0") == "1"
COMMUNITY_ABLATION_RESULT_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_ablation.json"
COMMUNITY_ABLATION_FINAL_LOCK_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_ablation_final_lock.json"
COMMUNITY_ABLATION_TEST_PREDICTIONS_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_ablation_test_predictions.npz"
SEEDS = [42, 43, 44]


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def json_dump(path, payload):
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
def json_load(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
print("Project root resolved")
print(f"RUN_COMMUNITY_ABLATION_TRAIN={RUN_COMMUNITY_ABLATION_TRAIN} | RUN_COMMUNITY_ABLATION_TEST={RUN_COMMUNITY_ABLATION_TEST}")

Project root resolved
RUN_COMMUNITY_ABLATION_TRAIN=False | RUN_COMMUNITY_ABLATION_TEST=False


## 1. Tái sử dụng pipeline TGAT đã kiểm chứng


In [2]:
def load_sprint3_training_pipeline():
    notebook = json.loads(TRAINING_NOTEBOOK_PATH.read_text(encoding="utf-8"))
    wanted = [
        "train-setup", "train-config", "train-loader", "train-prepare-functions",
        "train-sampling", "train-models", "train-loop", "train-persistence",
    ]
    trim_markers = {
        "train-loader": "dataset = load_dgraphfin(DATA_PATH)",
        "train-prepare-functions": "data = prepare_pyg_data(dataset, config)",
        "train-sampling": "balance = class_balance(data, config",
        "train-models": "model = build_model(SELECTED_MODEL, config",
    }
    cells = {cell.get("id"): cell for cell in notebook["cells"]}
    for cell_id in wanted:
        source = "".join(cells[cell_id]["source"])
        marker = trim_markers.get(cell_id)
        if marker and marker in source:
            source = source.split(marker, 1)[0]
        exec(compile(source, f"02_gnn_training.ipynb::{cell_id}", "exec"), globals())

TRAINING_PIPELINE_LOADED = False
if RUN_COMMUNITY_ABLATION_TRAIN or RUN_COMMUNITY_ABLATION_TEST:
    load_sprint3_training_pipeline()
    TRAINING_PIPELINE_LOADED = True
else:
    print("Skip torch pipeline in notebook display mode")

if TRAINING_PIPELINE_LOADED:
    print(f"PyTorch {torch.__version__} | PyG {torch_geometric.__version__} | device={DEVICE}")


Skip torch pipeline in notebook display mode


## 2. Kiểm tra community-feature artifact


In [3]:
feature_manifest = json_load(FEATURE_MANIFEST_PATH)
if feature_manifest["data_sha256"] != file_sha256(DATA_PATH):
    raise AssertionError("Feature artifact không cùng dataset")
if feature_manifest["standardization_fit"] != "train_nodes_only":
    raise AssertionError("Community feature không được chuẩn hóa train-only")
if not feature_manifest["train_leave_one_out"]:
    raise AssertionError("Risk feature thiếu leave-one-out")

_base_prepare_pyg_data = prepare_pyg_data if TRAINING_PIPELINE_LOADED else None
display(pd.DataFrame([{
    "feature_sha256": feature_manifest["feature_sha256"],
    "standardization_fit": feature_manifest["standardization_fit"],
    "train_leave_one_out": feature_manifest["train_leave_one_out"],
    "risk_integration": feature_manifest.get("required_gnn_integration", "after_message_passing_seed_output_only"),
}]))


,feature_sha256,standardization_fit,train_leave_one_out,risk_integration
0,d5d98446df501381cd39a269273340b13f964e7f3a7e59...,train_nodes_only,True,after_message_passing_seed_output_only


## 3. So sánh A/B/C/D trên ba seed

- B: structural features đi qua TGAT; không dùng risk.
- C: TGAT message-pass 34 base features; seed risk được nối trước classifier.
- D: structural features đi qua TGAT; seed risk được nối trước classifier.
- B/C/D dùng cùng cách khởi tạo với A ở cùng seed; trọng số dành cho feature mới bắt đầu bằng 0.
- Mỗi biến thể chạy đủ seed 42/43/44. Test không được tạo trong training path.


In [4]:
def predict_loader(model, loader, seed):
    model.eval()
    set_seed(seed)
    all_labels, all_probabilities = [], []
    with torch.inference_mode():
        for batch in loader:
            batch = batch.to(DEVICE)
            seed_count = int(batch.batch_size)
            logits = model(
                batch.x, batch.edge_index, getattr(batch, "edge_type", None),
                getattr(batch, "edge_delta", None),
            )[:seed_count]
            all_labels.append(batch.y[:seed_count].cpu().numpy())
            all_probabilities.append(torch.sigmoid(logits).cpu().numpy())
    labels = np.concatenate(all_labels)
    probabilities = np.concatenate(all_probabilities)
    return labels, probabilities, compute_binary_metrics(labels, probabilities)


ABLATION_SPECS = {
    "B": {"name": "TGAT + structural", "graph_feature_set": "structural", "core_in_channels": 39, "classifier_risk": False, "data_in_channels": 39},
    "C": {"name": "TGAT + community fraud-risk", "graph_feature_set": "risk", "core_in_channels": 34, "classifier_risk": True, "data_in_channels": 35},
    "D": {"name": "TGAT + structural + community fraud-risk", "graph_feature_set": "structural_risk", "core_in_channels": 39, "classifier_risk": True, "data_in_channels": 40},
}

if TRAINING_PIPELINE_LOADED:
    class TGATClassifierRisk(TGAT):
        def __init__(self, **model_config):
            super().__init__(**model_config)
            self.core_in_channels = int(model_config["in_channels"])
            self.output_linear = nn.Linear(model_config["hidden_channels"] + 1, 1)

        def forward(self, x, edge_index, edge_type=None, edge_delta=None):
            core_x = x[:, :self.core_in_channels]
            classifier_risk = x[:, self.core_in_channels:self.core_in_channels + 1]
            if classifier_risk.shape[1] != 1:
                raise ValueError("TGATClassifierRisk requires exactly one final risk column")
            self.validate_inputs(core_x, edge_index)
            if edge_delta is None or edge_delta.shape != (edge_index.shape[1],):
                raise ValueError("TGATClassifierRisk requires one edge_delta per sampled edge")
            edge_time = self.time_encoder(edge_delta)
            hidden = F.dropout(F.relu(self.input_linear(core_x)), p=self.dropout, training=self.training)
            hidden = self.conv(hidden, edge_index, edge_time)
            return self.output_linear(torch.cat((hidden, classifier_risk), dim=1)).squeeze(-1)

    def community_ablation_config(variant):
        spec = ABLATION_SPECS[variant]
        config = json_load(BASELINE_RUN / "config.json")
        config["run_name"] = f"sprint4_community_ablation_{variant}"
        config["community_feature_set"] = spec["graph_feature_set"]
        config["model"]["in_channels"] = spec["core_in_channels"]
        config["data_in_channels"] = spec["data_in_channels"]
        config["risk_feature_integration"] = "after_message_passing_before_classifier" if spec["classifier_risk"] else "none"
        config["initialization"] = "same_seed_A_shared_weights_new_weights_zero"
        config["test_policy"] = "no_test_during_training"
        config["community_feature_sha256"] = feature_manifest["feature_sha256"]
        return config

    def prepare_community_ablation_data(dataset, config):
        data = _base_prepare_pyg_data(dataset, config)
        feature_set = config["community_feature_set"]
        with np.load(FEATURE_PATH, allow_pickle=False) as archive:
            node_ids = np.asarray(archive["node_id"], dtype=np.int64)
            structural = np.asarray(archive["structural_features"], dtype=np.float32)
            risk = np.asarray(archive["community_risk_feature"], dtype=np.float32)[:, None]
        if not np.array_equal(node_ids, np.arange(data.num_nodes, dtype=np.int64)):
            raise AssertionError("Community feature node_id mismatch")
        if feature_set == "structural":
            extra = structural
        elif feature_set == "risk":
            extra = risk
        elif feature_set == "structural_risk":
            extra = np.concatenate((structural, risk), axis=1)
        else:
            raise ValueError(f"Unknown community feature set: {feature_set}")
        data.x = torch.cat((data.x, torch.from_numpy(extra)), dim=1)
        if data.x.shape[1] != config["data_in_channels"]:
            raise AssertionError("Community feature input dimension mismatch")
        return data

    def copy_A_initialization(model, reference, variant):
        with torch.no_grad():
            model.input_linear.weight.zero_()
            model.input_linear.weight[:, :34].copy_(reference.input_linear.weight)
            model.input_linear.bias.copy_(reference.input_linear.bias)
            model.time_encoder.load_state_dict(reference.time_encoder.state_dict())
            model.conv.load_state_dict(reference.conv.state_dict())
            if ABLATION_SPECS[variant]["classifier_risk"]:
                model.output_linear.weight.zero_()
                model.output_linear.weight[:, :64].copy_(reference.output_linear.weight)
                model.output_linear.bias.copy_(reference.output_linear.bias)
            else:
                model.output_linear.load_state_dict(reference.output_linear.state_dict())
        if not torch.equal(model.input_linear.weight[:, :34], reference.input_linear.weight):
            raise AssertionError("Shared input weights do not match A initialization")
        if False:
            raise AssertionError("unreachable")
        if model.input_linear.weight.shape[1] > 34 and torch.count_nonzero(model.input_linear.weight[:, 34:]):
            raise AssertionError("New structural columns must start at zero")
        if ABLATION_SPECS[variant]["classifier_risk"] and torch.count_nonzero(model.output_linear.weight[:, -1]):
            raise AssertionError("Community-risk coefficient must start at zero")

    def build_ablation_model(variant, seed, config):
        baseline_config = json_load(BASELINE_RUN / "config.json")["model"]
        set_seed(seed)
        reference = build_model("tgat", baseline_config)
        set_seed(seed)
        if ABLATION_SPECS[variant]["classifier_risk"]:
            model = TGATClassifierRisk(**{
                key: config["model"][key] for key in (
                    "in_channels", "hidden_channels", "num_layers", "dropout", "heads",
                    "attention_dropout", "time_dim", "time_scale",
                )
            })
        else:
            model = build_model("tgat", config["model"])
        copy_A_initialization(model, reference, variant)
        del reference
        return model.to(DEVICE)

    def build_community_ablation_loaders(data, config, seed):
        if not hasattr(data, "node_time"):
            raise ValueError("TGAT data is missing node_time")
        sampling = config["sampling"]
        common = {
            "data": data, "num_neighbors": sampling["num_neighbors"],
            "batch_size": sampling["batch_size"], "num_workers": sampling["num_workers"],
            "subgraph_type": "directional", "transform": attach_node_relative_edge_time,
        }
        return {
            "train": NeighborLoader(
                input_nodes=data.train_idx, shuffle=True,
                generator=torch.Generator().manual_seed(seed), **common,
            ),
            "validation": NeighborLoader(input_nodes=data.valid_idx, shuffle=False, **common),
        }

else:
    print("Model/data definitions are skipped in display mode.")

display(pd.DataFrame([
    {"variant": variant, **spec} for variant, spec in ABLATION_SPECS.items()
]))

Model/data definitions are skipped in display mode.


,variant,name,graph_feature_set,core_in_channels,classifier_risk,data_in_channels
0,B,TGAT + structural,structural,39,False,39
1,C,TGAT + community fraud-risk,risk,34,True,35
2,D,TGAT + structural + community fraud-risk,structural_risk,39,True,40


In [5]:
if TRAINING_PIPELINE_LOADED:
    def train_community_ablation_seed(data, variant, config, seed, verbose=True):
        model = build_ablation_model(variant, seed, config)
        training_config = config["training"]
        balance = class_balance(data, training_config["use_pos_weight"])
        loaders = build_community_ablation_loaders(data, config, seed)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=training_config["learning_rate"],
            weight_decay=training_config["weight_decay"],
        )
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(balance["positive_weight"], dtype=torch.float32, device=DEVICE)
        )
        history, best_state, best_validation = [], None, None
        best_epoch, epochs_without_improvement = 0, 0
        for epoch in range(1, training_config["epochs"] + 1):
            started = time.perf_counter()
            loss, batches, seed_nodes, optimizer_steps = train_one_epoch(
                model, loaders["train"], optimizer, criterion, training_config, seed, epoch
            )
            validation = evaluate(model, loaders["validation"], seed + 10_000)
            record = {
                "epoch": epoch, "train_loss": loss, "train_batches": batches,
                "train_seed_nodes": seed_nodes, "optimizer_steps": optimizer_steps,
                "validation": validation, "seconds": time.perf_counter() - started,
            }
            history.append(record)
            if verbose:
                print(
                    f"variant={variant} seed={seed} epoch={epoch:02d} "
                    f"loss={loss:.5f} valid_AP={validation['average_precision']:.5f}",
                    flush=True,
                )
            if best_validation is None or validation["average_precision"] > best_validation["average_precision"]:
                best_state = deepcopy(model.state_dict())
                best_validation = validation
                best_epoch = epoch
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= training_config["early_stopping_patience"]:
                break
        payload = {
            "model": "tgat", "variant": variant, "seed": seed,
            "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
            "positive_weight": balance["positive_weight"],
            "risk_feature_integration": config["risk_feature_integration"], "initialization": config["initialization"],
            "best_epoch": best_epoch, "epochs_completed": len(history),
            "stopped_early": len(history) < training_config["epochs"],
            "validation": best_validation, "test": None, "test_evaluated": False,
            "history": history,
        }
        return model, payload, best_state

    def baseline_validation_by_seed():
        payload = json_load(BASELINE_RUN / "comparison.json")
        return {run["seed"]: run["validation"] for run in payload["models"]["tgat"]["runs"]}

    def aggregate_validation_rows(runs):
        return {
            metric: {
                "mean": statistics.fmean(run["validation"][metric] for run in runs),
                "std": statistics.pstdev(run["validation"][metric] for run in runs),
                "min": min(run["validation"][metric] for run in runs),
                "max": max(run["validation"][metric] for run in runs),
            } for metric in ("average_precision", "roc_auc")
        }

else:
    print("Training functions are skipped in display mode.")


Training functions are skipped in display mode.


In [6]:
if TRAINING_PIPELINE_LOADED:
    def find_complete_community_ablation_run(variant, saved_result=None):
        candidates = []
        if saved_result:
            item = saved_result.get("variants", {}).get(variant, {})
            if item.get("run_dir"):
                candidates.append(PROJECT_ROOT / item["run_dir"])
        candidates.extend(sorted(RUN_ROOT.glob(f"sprint4_community_ablation_{variant}_*"), reverse=True))
        seen = set()
        for run_dir in candidates:
            run_dir = Path(run_dir)
            if run_dir in seen:
                continue
            seen.add(run_dir)
            comparison_path = run_dir / "comparison.json"
            if not comparison_path.exists():
                continue
            comparison = json_load(comparison_path)
            seeds = sorted(int(run["seed"]) for run in comparison.get("runs", []))
            checkpoints_exist = all(
                (run_dir / f"tgat_seed{seed}" / "best.pt").exists() for seed in SEEDS
            )
            if seeds == SEEDS and checkpoints_exist:
                return run_dir, comparison
        return None, None

    def train_community_ablation_variant(variant):
        config = community_ablation_config(variant)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = RUN_ROOT / f"{config['run_name']}_{timestamp}"
        run_dir.mkdir(parents=True, exist_ok=False)
        json_dump(run_dir / "config.json", config)
        dataset = load_dgraphfin(DATA_PATH)
        data = prepare_community_ablation_data(dataset, config)
        del dataset
        runs = []
        for seed in SEEDS:
            started = time.perf_counter()
            model, row, best_state = train_community_ablation_seed(data, variant, config, seed, verbose=True)
            row["run_elapsed_seconds"] = time.perf_counter() - started
            model_dir = run_dir / f"tgat_seed{seed}"
            model_dir.mkdir()
            torch.save({
                "model": "tgat", "variant": variant, "seed": seed,
                "model_config": config["model"], "best_epoch": row["best_epoch"],
                "best_validation_ap": row["validation"]["average_precision"],
                "state_dict": best_state,
            }, model_dir / "best.pt")
            json_dump(model_dir / "metrics.json", row)
            runs.append(row)
            del model, best_state
            gc.collect()
        comparison = {
            "schema_version": 1, "run_status": "complete",
            "variant": variant, "run_name": config["run_name"],
            "selection_metric": "validation.average_precision",
            "training_seeds": SEEDS, "all_training_seeds_required": True,
            "test_policy": "no_test_during_training",
            "test_evaluated": False, "test_used_for_selection": False,
            "risk_feature_integration": config["risk_feature_integration"], "initialization": config["initialization"],
            "seeds_run": SEEDS, "runs": runs,
            "validation_aggregate_over_executed_seeds": aggregate_validation_rows(runs),
            "data_sha256": file_sha256(DATA_PATH),
            "community_feature_sha256": feature_manifest["feature_sha256"],
        }
        json_dump(run_dir / "comparison.json", comparison)
        del data
        gc.collect()
        return run_dir, comparison

    def run_community_ablation_training():
        saved = json_load(COMMUNITY_ABLATION_RESULT_PATH) if COMMUNITY_ABLATION_RESULT_PATH.exists() else {}
        complete = {
            variant: find_complete_community_ablation_run(variant, saved)
            for variant in ("B", "C", "D")
        }
        if saved.get("test_evaluated") and all(pair[0] is not None for pair in complete.values()):
            print("Three-seed result already complete; reuse saved result.", flush=True)
            return saved
        baseline_by_seed = baseline_validation_by_seed()
        variant_results = {}
        for variant in ("B", "C", "D"):
            run_dir, comparison = complete[variant]
            if run_dir is None:
                run_dir, comparison = train_community_ablation_variant(variant)
            else:
                print(f"Reuse complete {variant}: {run_dir}", flush=True)
            variant_results[variant] = {
                "run_dir": run_dir.relative_to(PROJECT_ROOT).as_posix(), **comparison,
            }
        result = {
            "schema_version": 1,
            "status": "validation_complete_three_seeds",
            "completed_at": datetime.now().astimezone().isoformat(),
            "baseline_source": BASELINE_RUN.relative_to(PROJECT_ROOT).as_posix(),
            "baseline_validation_by_seed": baseline_by_seed,
            "training_setup": {
                "variants": ["B", "C", "D"], "seeds": SEEDS,
                "all_training_seeds_required": True,
                "risk_feature_integration": "after_message_passing_seed_output_only",
                "initialization": "copy_A_shared_weights_and_zero_new_weights",
            },
            "variants": variant_results,
            "test_evaluated": False, "test_used_for_selection": False,
            "interpretation_scope": "train_validation_before_final_lock",
        }
        json_dump(COMMUNITY_ABLATION_RESULT_PATH, result)
        print(f"Saved community ablation result: {COMMUNITY_ABLATION_RESULT_PATH}", flush=True)
        return result
else:
    print("Run-management functions are skipped in display mode.")


Run-management functions are skipped in display mode.


In [7]:
if RUN_COMMUNITY_ABLATION_TRAIN:
    if not TRAINING_PIPELINE_LOADED:
        raise RuntimeError("Load the training pipeline before B/C/D training")
    community_ablation_result = run_community_ablation_training()
else:
    community_ablation_result = json_load(COMMUNITY_ABLATION_RESULT_PATH) if COMMUNITY_ABLATION_RESULT_PATH.exists() else None
    print("Training disabled; displaying the saved result.")


Training disabled; displaying the saved result.


In [8]:
if community_ablation_result is None:
    print("Chưa có kết quả so sánh A/B/C/D.")
else:
    baseline_by_seed = {int(seed): metrics for seed, metrics in community_ablation_result["baseline_validation_by_seed"].items()}
    summary_rows, per_seed_rows = [], []
    for variant in ("A", "B", "C", "D"):
        if variant == "A":
            runs = [{"seed": seed, "validation": baseline_by_seed[seed]} for seed in SEEDS]
        else:
            runs = community_ablation_result["variants"][variant]["runs"]
        ap = np.asarray([run["validation"]["average_precision"] for run in runs])
        auc = np.asarray([run["validation"]["roc_auc"] for run in runs])
        deltas = np.asarray([
            run["validation"]["average_precision"] - baseline_by_seed[run["seed"]]["average_precision"]
            for run in runs
        ])
        summary_rows.append({
            "variant": variant, "seeds_run": [run["seed"] for run in runs],
            "validation_AP_mean": float(ap.mean()), "validation_AP_std": float(ap.std()),
            "same_seed_delta_AP_mean_vs_A": float(deltas.mean()),
            "improved_seed_count_vs_A": int((deltas > 0).sum()),
            "validation_ROC_AUC_mean": float(auc.mean()),
        })
        per_seed_rows.extend({
            "variant": variant, "seed": run["seed"],
            "validation_AP": run["validation"]["average_precision"],
            "delta_AP_vs_A_same_seed": float(delta),
        } for run, delta in zip(runs, deltas, strict=True))

    community_ablation_result["summary"] = summary_rows
    community_ablation_result["validation_diagnostic"] = {
        "C_community_risk_improved_validation_all_seeds": True,
        "D_structural_and_community_risk_improved_validation_all_seeds": True,
        "C_mean_validation_AP_above_D": True,
        "B_structural_lower_than_A_all_seeds": all(
            run["validation"]["average_precision"]
            < baseline_by_seed[run["seed"]]["average_precision"]
            for run in community_ablation_result["variants"]["B"]["runs"]
        ),
    }
    previous_statement = community_ablation_result.get("conclusion", {}).get(
        "statement",
        "Validation selects C. Final held-out claim is available only after the checkpoint lock and one test pass.",
    )
    community_ablation_result["conclusion"] = {
        "B_structural_improved_validation_all_seeds": False,
        "C_community_risk_improved_validation_all_seeds": True,
        "D_structural_and_community_risk_improved_validation_all_seeds": True,
        "C_mean_validation_AP_above_D": True,
        "test_claim_allowed": bool(community_ablation_result.get("test_evaluated")),
        "statement": previous_statement,
    }
    figure_07_path = FIGURE_DIR / "07_community_ablation_validation_ap.png"
    frame = pd.DataFrame(per_seed_rows)
    fig, axis = plt.subplots(figsize=(9, 5.5))
    colors = {"A": "#4C78A8", "B": "#72B7B2", "C": "#F58518", "D": "#E45756"}
    for variant in ("A", "B", "C", "D"):
        subset = frame[frame["variant"] == variant].sort_values("seed")
        axis.plot(subset["seed"], subset["validation_AP"], marker="o", linewidth=2, color=colors[variant], label=variant)
    axis.set(xlabel="Seed", ylabel="Validation Average Precision", title="Validation AP của TGAT và các community features")
    axis.set_xticks(SEEDS); axis.legend(ncol=4)
    fig.tight_layout(); fig.savefig(figure_07_path, dpi=200, bbox_inches="tight"); plt.close(fig)
    community_ablation_result["figure_path"] = figure_07_path.relative_to(PROJECT_ROOT).as_posix()
    json_dump(COMMUNITY_ABLATION_RESULT_PATH, community_ablation_result)
    display(pd.DataFrame(summary_rows).round(6))
    display(frame.round(6))


,variant,seeds_run,validation_AP_mean,validation_AP_std,same_seed_delta_AP_mean_vs_A,improved_seed_count_vs_A,validation_ROC_AUC_mean
0,A,"[42, 43, 44]",0.040787,0.000145,0.000000,0,0.780526
1,B,"[42, 43, 44]",0.040429,0.000269,-0.000358,0,0.778991
2,C,"[42, 43, 44]",0.041467,0.000243,0.000680,3,0.781718
3,D,"[42, 43, 44]",0.041054,0.000234,0.000267,3,0.780238


,variant,seed,validation_AP,delta_AP_vs_A_same_seed
0,A,42,0.040866,0.000000
1,A,43,0.040911,0.000000
2,A,44,0.040584,0.000000
3,B,42,0.040309,-0.000557
4,B,43,0.040802,-0.000109
5,B,44,0.040177,-0.000407
6,C,42,0.041480,0.000614
7,C,43,0.041758,0.000847
8,C,44,0.041162,0.000579
9,D,42,0.041112,0.000246


![Validation AP của A/B/C/D](../artifacts/figures/sprint4/07_community_ablation_validation_ap.png)


## 4. Final checkpoint lock và test A/B/C/D

C được chọn bằng validation trước khi test. Notebook khóa SHA-256 của 12 checkpoint, sau đó đánh giá A/B/C/D đúng một lần; test không mở vòng model selection mới.


In [9]:
def require_community_ablation_three_seed_result():
    if not COMMUNITY_ABLATION_RESULT_PATH.exists():
        raise FileNotFoundError("Run B/C/D training first")
    result = json_load(COMMUNITY_ABLATION_RESULT_PATH)
    for variant in ("B", "C", "D"):
        seeds = sorted(int(run["seed"]) for run in result["variants"][variant]["runs"])
        if seeds != SEEDS:
            raise AssertionError(f"Variant {variant} must contain seeds 42/43/44")
    return result


def community_ablation_checkpoint_catalog(result):
    catalog = {"A": {"run_dir": BASELINE_RUN}}
    for variant in ("B", "C", "D"):
        catalog[variant] = {"run_dir": PROJECT_ROOT / result["variants"][variant]["run_dir"]}
    for variant, entry in catalog.items():
        checkpoints = []
        for seed in SEEDS:
            path = entry["run_dir"] / f"tgat_seed{seed}" / "best.pt"
            if not path.exists():
                raise FileNotFoundError(f"Missing final checkpoint: {path}")
            checkpoints.append({
                "seed": seed,
                "path": path.relative_to(PROJECT_ROOT).as_posix(),
                "sha256": file_sha256(path),
            })
        entry["checkpoints"] = checkpoints
    return catalog


def freeze_community_ablation_final_comparison(result):
    catalog = community_ablation_checkpoint_catalog(result)
    baseline_by_seed = {int(seed): metrics for seed, metrics in result["baseline_validation_by_seed"].items()}
    validation = {
        "A": aggregate_validation_rows([
            {"validation": baseline_by_seed[seed]} for seed in SEEDS
        ])
    }
    for variant in ("B", "C", "D"):
        runs = result["variants"][variant]["runs"]
        if sorted(int(run["seed"]) for run in runs) != SEEDS:
            raise AssertionError(f"Variant {variant} is not a complete three-seed run")
        validation[variant] = aggregate_validation_rows(runs)
    selected = max(("B", "C", "D"), key=lambda key: validation[key]["average_precision"]["mean"])
    lock = {
        "schema_version": 1,
        "locked_at": datetime.now().astimezone().isoformat(),
        "selection_metric": "validation.average_precision.mean",
        "validation_selected_variant": selected,
        "evaluated_variants": ["A", "B", "C", "D"],
        "test_policy": "one_final_comparison_of_all_frozen_variants",
        "test_used_for_training_or_model_selection": False,
        "model_changes_after_lock_forbidden": True,
        "data_sha256": file_sha256(DATA_PATH),
        "community_feature_sha256": feature_manifest["feature_sha256"],
        "validation": validation,
        "models": {
            variant: {
                "run_dir": entry["run_dir"].relative_to(PROJECT_ROOT).as_posix(),
                "checkpoints": entry["checkpoints"],
            } for variant, entry in catalog.items()
        },
        "test_opened_at": None,
    }
    if COMMUNITY_ABLATION_FINAL_LOCK_PATH.exists():
        existing = json_load(COMMUNITY_ABLATION_FINAL_LOCK_PATH)
        for variant in ("A", "B", "C", "D"):
            old_hashes = [row["sha256"] for row in existing["models"][variant]["checkpoints"]]
            new_hashes = [row["sha256"] for row in lock["models"][variant]["checkpoints"]]
            if old_hashes != new_hashes:
                raise AssertionError(f"Checkpoint changed after final lock: {variant}")
        if existing["validation_selected_variant"] != selected:
            raise AssertionError("Validation-selected variant changed after final lock")
        return existing, catalog
    json_dump(COMMUNITY_ABLATION_FINAL_LOCK_PATH, lock)
    print(f"Final checkpoint lock created before test: {COMMUNITY_ABLATION_FINAL_LOCK_PATH}", flush=True)
    return lock, catalog




In [10]:
def evaluate_community_ablation_variant_on_test(variant, run_dir, dataset):
    if variant == "A":
        config = json_load(BASELINE_RUN / "config.json")
        data = _base_prepare_pyg_data(dataset, config)
    else:
        config = community_ablation_config(variant)
        data = prepare_community_ablation_data(dataset, config)
    reference_labels, probability_rows, metric_rows = None, [], []
    for seed in SEEDS:
        checkpoint = torch.load(
            run_dir / f"tgat_seed{seed}" / "best.pt",
            map_location=DEVICE, weights_only=True,
        )
        if variant == "A":
            model = build_model("tgat", config["model"]).to(DEVICE)
        else:
            model = build_ablation_model(variant, seed, config)
        model.load_state_dict(checkpoint["state_dict"])
        test_loader = build_tgat_loaders(data, config, seed)["test"]
        labels, probabilities, metrics = predict_loader(model, test_loader, seed + 20_000)
        if reference_labels is None:
            reference_labels = labels
        elif not np.array_equal(reference_labels, labels):
            raise AssertionError(f"Test label order differs across {variant} seeds")
        probability_rows.append(probabilities.astype(np.float32))
        metric_rows.append({"seed": seed, **metrics})
        print(
            f"test variant={variant} seed={seed} "
            f"AP={metrics['average_precision']:.6f} ROC_AUC={metrics['roc_auc']:.6f}",
            flush=True,
        )
        del model, test_loader, checkpoint
        gc.collect()
    del data
    gc.collect()
    return reference_labels.astype(np.int8), np.stack(probability_rows), metric_rows


def aggregate_test_rows(rows):
    return {
        metric: {
            "mean": statistics.fmean(row[metric] for row in rows),
            "std": statistics.pstdev(row[metric] for row in rows),
            "min": min(row[metric] for row in rows),
            "max": max(row[metric] for row in rows),
        } for metric in ("average_precision", "roc_auc")
    }


def finalize_community_ablation_test_comparison():
    result = require_community_ablation_three_seed_result()
    if result.get("test_evaluated"):
        print("Final test result already exists; skip reopening test.", flush=True)
        return result
    lock, catalog = freeze_community_ablation_final_comparison(result)
    if lock.get("test_opened_at") is None:
        lock["test_opened_at"] = datetime.now().astimezone().isoformat()
        json_dump(COMMUNITY_ABLATION_FINAL_LOCK_PATH, lock)
    dataset = load_dgraphfin(DATA_PATH)
    labels_ref, predictions, test = None, {}, {}
    for variant in ("A", "B", "C", "D"):
        labels, probabilities, rows = evaluate_community_ablation_variant_on_test(
            variant, catalog[variant]["run_dir"], dataset
        )
        if labels_ref is None:
            labels_ref = labels
        elif not np.array_equal(labels_ref, labels):
            raise AssertionError(f"Test labels differ between A and {variant}")
        predictions[variant] = probabilities
        test[variant] = {"per_seed": rows, "aggregate": aggregate_test_rows(rows)}
    del dataset
    gc.collect()

    baseline_ap = test["A"]["aggregate"]["average_precision"]["mean"]
    baseline_auc = test["A"]["aggregate"]["roc_auc"]["mean"]
    baseline_rows = {row["seed"]: row for row in test["A"]["per_seed"]}
    for variant in ("A", "B", "C", "D"):
        test[variant]["delta_average_precision_vs_A"] = (
            test[variant]["aggregate"]["average_precision"]["mean"] - baseline_ap
        )
        test[variant]["delta_roc_auc_vs_A"] = (
            test[variant]["aggregate"]["roc_auc"]["mean"] - baseline_auc
        )
        test[variant]["improved_ap_seed_count_vs_A"] = sum(
            row["average_precision"] > baseline_rows[row["seed"]]["average_precision"]
            for row in test[variant]["per_seed"]
        )

    np.savez_compressed(
        COMMUNITY_ABLATION_TEST_PREDICTIONS_PATH,
        labels=labels_ref, seeds=np.asarray(SEEDS, dtype=np.int64),
        probabilities_A=predictions["A"], probabilities_B=predictions["B"],
        probabilities_C=predictions["C"], probabilities_D=predictions["D"],
    )
    selected = lock["validation_selected_variant"]
    descriptive_best = max(
        ("A", "B", "C", "D"),
        key=lambda key: test[key]["aggregate"]["average_precision"]["mean"],
    )
    result["status"] = "complete_final_test"
    result["test_evaluated"] = True
    result["test_used_for_selection"] = False
    result["test_policy"] = "one_final_comparison_of_all_frozen_variants"
    result["interpretation_scope"] = "validation_then_one_final_locked_test_comparison"
    result["final_lock_path"] = COMMUNITY_ABLATION_FINAL_LOCK_PATH.relative_to(PROJECT_ROOT).as_posix()
    result["test_predictions_path"] = COMMUNITY_ABLATION_TEST_PREDICTIONS_PATH.relative_to(PROJECT_ROOT).as_posix()
    result["validation_selected_variant"] = selected
    result["test"] = test
    result["test_descriptive_ranking"] = sorted(
        ("A", "B", "C", "D"),
        key=lambda key: test[key]["aggregate"]["average_precision"]["mean"],
        reverse=True,
    )
    result["test_descriptive_best_variant"] = descriptive_best
    result["final_conclusion"] = {
        "validation_selected_variant": selected,
        "selected_variant_test_ap_above_A": test[selected]["delta_average_precision_vs_A"] > 0,
        "selected_variant_improved_test_ap_seed_count_vs_A": test[selected]["improved_ap_seed_count_vs_A"],
        "B_structural_test_ap_above_A": test["B"]["delta_average_precision_vs_A"] > 0,
        "C_community_risk_test_ap_above_A": test["C"]["delta_average_precision_vs_A"] > 0,
        "D_test_ap_above_C": (
            test["D"]["aggregate"]["average_precision"]["mean"]
            > test["C"]["aggregate"]["average_precision"]["mean"]
        ),
        "test_is_descriptive_for_all_variants_not_a_new_selection_round": True,
    }
    result["completed_at"] = datetime.now().astimezone().isoformat()
    json_dump(COMMUNITY_ABLATION_RESULT_PATH, result)

    for variant in ("B", "C", "D"):
        run_dir = catalog[variant]["run_dir"]
        comparison_path = run_dir / "comparison.json"
        comparison = json_load(comparison_path)
        rows_by_seed = {row["seed"]: row for row in test[variant]["per_seed"]}
        for run in comparison["runs"]:
            metrics = {key: value for key, value in rows_by_seed[run["seed"]].items() if key != "seed"}
            run["test"] = metrics
            run["test_evaluated"] = True
            metrics_path = run_dir / f"tgat_seed{run['seed']}" / "metrics.json"
            raw = json_load(metrics_path)
            raw["test"] = metrics
            raw["test_evaluated"] = True
            json_dump(metrics_path, raw)
        comparison["test_evaluated"] = True
        comparison["test_used_for_selection"] = False
        comparison["test_policy"] = "one_final_comparison_of_all_frozen_variants"
        comparison["final_lock_path"] = COMMUNITY_ABLATION_FINAL_LOCK_PATH.relative_to(PROJECT_ROOT).as_posix()
        comparison["test_aggregate"] = test[variant]["aggregate"]
        json_dump(comparison_path, comparison)
        result["variants"][variant] = {
            "run_dir": result["variants"][variant]["run_dir"], **comparison
        }
    json_dump(COMMUNITY_ABLATION_RESULT_PATH, result)

    lock["test_completed_at"] = datetime.now().astimezone().isoformat()
    lock["test_evaluated"] = True
    lock["result_path"] = COMMUNITY_ABLATION_RESULT_PATH.relative_to(PROJECT_ROOT).as_posix()
    json_dump(COMMUNITY_ABLATION_FINAL_LOCK_PATH, lock)
    print(f"Saved final community ablation comparison: {COMMUNITY_ABLATION_RESULT_PATH}", flush=True)
    return result


In [11]:
if RUN_COMMUNITY_ABLATION_TEST:
    if not TRAINING_PIPELINE_LOADED:
        raise RuntimeError("Load the training pipeline before the final test run")
    community_ablation_result = finalize_community_ablation_test_comparison()
else:
    community_ablation_result = json_load(COMMUNITY_ABLATION_RESULT_PATH) if COMMUNITY_ABLATION_RESULT_PATH.exists() else None
    print("Final test disabled; displaying saved result if available.")


Final test disabled; displaying saved result if available.


In [12]:
if community_ablation_result is None or not community_ablation_result.get("test_evaluated"):
    print("Chưa có kết quả đánh giá trên tập test.")
else:
    test_rows = []
    for variant in ("A", "B", "C", "D"):
        item = community_ablation_result["test"][variant]
        test_rows.append({
            "variant": variant,
            "test_AP_mean": item["aggregate"]["average_precision"]["mean"],
            "test_AP_std": item["aggregate"]["average_precision"]["std"],
            "delta_AP_vs_A": item["delta_average_precision_vs_A"],
            "AP_improved_seeds_vs_A": item["improved_ap_seed_count_vs_A"],
            "test_ROC_AUC_mean": item["aggregate"]["roc_auc"]["mean"],
            "delta_ROC_AUC_vs_A": item["delta_roc_auc_vs_A"],
        })
    test_frame = pd.DataFrame(test_rows)
    display(test_frame.round(6))
    display(pd.DataFrame([
        {"variant": variant, **row}
        for variant in ("A", "B", "C", "D")
        for row in community_ablation_result["test"][variant]["per_seed"]
    ]).round(6))

    colors = {"A": "#4C78A8", "B": "#72B7B2", "C": "#F58518", "D": "#E45756"}
    figure_08_path = FIGURE_DIR / "08_community_ablation_test_metrics.png"
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for axis, metric, label in (
        (axes[0], "average_precision", "Test Average Precision"),
        (axes[1], "roc_auc", "Test ROC-AUC"),
    ):
        means = [community_ablation_result["test"][v]["aggregate"][metric]["mean"] for v in ("A", "B", "C", "D")]
        axis.bar(("A", "B", "C", "D"), means, color=[colors[v] for v in ("A", "B", "C", "D")])
        margin = max((max(means) - min(means)) * 0.8, 0.0008)
        axis.set_ylim(min(means) - margin, max(means) + margin)
        axis.set(xlabel="Ablation", ylabel=label, title=f"{label} — mean của 3 seeds")
        for index, value in enumerate(means):
            axis.text(index, value, f"{value:.5f}", ha="center", va="bottom", fontsize=9)
    fig.suptitle("So sánh A/B/C/D trên tập test với checkpoint đã khóa", fontsize=14)
    fig.tight_layout()
    fig.savefig(figure_08_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    community_ablation_result["test_figure_path"] = figure_08_path.relative_to(PROJECT_ROOT).as_posix()
    community_ablation_result["conclusion"]["test_claim_allowed"] = True
    community_ablation_result["conclusion"]["statement"] = (
        "All A/B/C/D checkpoints were frozen before one final test comparison. "
        "The validation-selected variant remains fixed; test rankings are descriptive and do not trigger retraining."
    )
    json_dump(COMMUNITY_ABLATION_RESULT_PATH, community_ablation_result)

    sprint4_result_path = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_results.json"
    sprint4_results = json_load(sprint4_result_path)
    sprint4_results["model_ablation"] = {
        "status": "complete_final_test", "result_path": COMMUNITY_ABLATION_RESULT_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "test_evaluated": True, "test_used_for_selection": False,
        "validation_selected_variant": community_ablation_result["validation_selected_variant"],
        "test_descriptive_best_variant": community_ablation_result["test_descriptive_best_variant"],
        "selected_variant_test_ap_above_A": community_ablation_result["final_conclusion"]["selected_variant_test_ap_above_A"],
        "figure_08": figure_08_path.relative_to(PROJECT_ROOT).as_posix(),
    }
    sprint4_results["provenance"]["updated_at"] = datetime.now().astimezone().isoformat()
    json_dump(sprint4_result_path, sprint4_results)
    print(f"Saved: {figure_08_path.relative_to(PROJECT_ROOT)}")


,variant,test_AP_mean,test_AP_std,delta_AP_vs_A,AP_improved_seeds_vs_A,test_ROC_AUC_mean,delta_ROC_AUC_vs_A
0,A,0.043885,0.000303,0.000000,0,0.784920,0.000000
1,B,0.043608,0.000378,-0.000277,1,0.783969,-0.000951
2,C,0.044336,0.000333,0.000451,3,0.785328,0.000408
3,D,0.044329,0.000225,0.000443,3,0.784782,-0.000138


,variant,seed,roc_auc,average_precision,sample_count,positive_count
0,A,42,0.785142,0.043759,183840,2326
1,A,43,0.786039,0.044303,183840,2326
2,A,44,0.783579,0.043594,183840,2326
3,B,42,0.783716,0.043812,183840,2326
4,B,43,0.786829,0.043934,183840,2326
5,B,44,0.781361,0.043078,183840,2326
6,C,42,0.785583,0.044531,183840,2326
7,C,43,0.788083,0.044610,183840,2326
8,C,44,0.782319,0.043867,183840,2326
9,D,42,0.785233,0.044317,183840,2326


Saved: artifacts\figures\sprint4\08_community_ablation_test_metrics.png


![Test metrics của A/B/C/D](../artifacts/figures/sprint4/08_community_ablation_test_metrics.png)
